In [ ]:
!pip install docx

In [ ]:
!pip install --upgrade python-docx

In [1]:
#Step1
import pandas as pd
import pandasql as ps
import pixiedust
import sys
sys.path.append('..')
import util
import etl
import pyarrow.parquet as pq
import pyarrow as pa
!pip install duckdb
import duckdb
import inspect

# Get the path of the imported module (etl.py)
etl_module_path = inspect.getfile(etl)
print("Path of the imported ETL module (etl.py):", etl_module_path)
util.usedatabase(spark, "real_world_data_jun_2022")

pixiedust.enableJobMonitor()

con = duckdb.connect()

Pixiedust database opened successfully


Please see https://github.com/pypa/pip/issues/5599 for advice on fixing the underlying issue.
To avoid this problem you can invoke Python with '-m pip' instead of running pip directly.
Path of the imported ETL module (etl.py): /home/o_suchsi/work/Oklahoma State/Priya/epilepsy/etl.py
Using real_world_data_jun_2022 ....
Successfully enabled Spark Job Progress Monitor


In [17]:
# Read the .txt file with a specific delimiter (e.g., tab-separated)
df = spark.read.option("delimiter", "\t").csv("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/package.txt")
df.show()
# Convert the Spark DataFrame to a Pandas DataFrame
pandas_df = df.toPandas()

# Show the contents of the Pandas DataFrame
print(pandas_df)

▸,:,


<IPython.core.display.Javascript object>

+--------------------+----------+--------------+--------------------+------------------+----------------+----------------+--------------+
|                 _c0|       _c1|           _c2|                 _c3|               _c4|             _c5|             _c6|           _c7|
+--------------------+----------+--------------+--------------------+------------------+----------------+----------------+--------------+
|           PRODUCTID|PRODUCTNDC|NDCPACKAGECODE|  PACKAGEDESCRIPTION|STARTMARKETINGDATE|ENDMARKETINGDATE|NDC_EXCLUDE_FLAG|SAMPLE_PACKAGE|
|0002-0213_458ef2a...| 0002-0213|  0002-0213-01|1 VIAL, MULTI-DOS...|          20230620|            null|               N|             N|
|0002-0800_dec32ea...| 0002-0800|  0002-0800-01|1 VIAL in 1 CARTO...|          19870710|            null|               N|             N|
|0002-1152_b597917...| 0002-1152|  0002-1152-01|1 VIAL, SINGLE-DO...|          20230728|            null|               N|             N|
|0002-1200_7832a4c...| 0002-1200| 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

                                                   _c0         _c1  \
0                                            PRODUCTID  PRODUCTNDC   
1       0002-0213_458ef2aa-cd5f-48bc-8829-82420cfed33b   0002-0213   
2       0002-0800_dec32ead-837e-4331-ab55-f3bbccea5b38   0002-0800   
3       0002-1152_b597917f-9673-4331-809d-79a538bb943c   0002-1152   
4       0002-1200_7832a4ce-bb0e-4753-a72c-a5bc9621f08c   0002-1200   
...                                                ...         ...   
215999  99528-606_f1e9f234-cca3-876f-e053-2995a90a5d40   99528-606   
216000  99528-606_f1e9f234-cca3-876f-e053-2995a90a5d40   99528-606   
216001  99528-606_f1e9f234-cca3-876f-e053-2995a90a5d40   99528-606   
216002  99528-606_f1e9f234-cca3-876f-e053-2995a90a5d40   99528-606   
216003  99528-606_f1e9f234-cca3-876f-e053-2995a90a5d40   99528-606   

                   _c2                                                _c3  \
0       NDCPACKAGECODE                                 PACKAGEDESCRIPTION   
1    

In [18]:
# Export the Pandas DataFrame to a CSV file
output_csv_path = "/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/ndctext.csv"
pandas_df.to_csv(output_csv_path, index=False)  # Set index=False to exclude the index column

▸,:,


In [19]:
df1 = spark.read.option("delimiter", "\t").csv("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/ndctext.csv")
display(df1)

In [2]:
#After Pairing-2
Epilepsy_Cohort_Paired_New = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/cohort_medication_Paired")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
#After Pairing-2
Epilepsy_Control_Paired_New = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/control_medication_Paired")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
#After Pairing-1
Epilepsy_Cohort_Paired_New1 = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/cohort_demo_paired_Final")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
Epilepsy_Cohort_Paired_New.printSchema()
Epilepsy_Control_Paired_New.printSchema()
# Epilepsy_Cohort_Paired_New1.printSchema()
# df = Epilepsy_Cohort_Paired_New.toPandas()
# display(df.head(10))

▸,:,


root
 |-- personid: string (nullable = true)
 |-- drugname: string (nullable = true)

root
 |-- personid: string (nullable = true)
 |-- drugname: string (nullable = true)



In [5]:
Epilepsy_Cohort_Paired_New.createOrReplaceTempView("Epilepsy_Cohort_Pair")
Epilepsy_Control_Paired_New.createOrReplaceTempView("Epilepsy_Control_Pair")
# Epilepsy_Cohort_Paired_New1.createOrReplaceTempView("Epilepsy_Cohort_Pair1")

▸,:,


In [23]:
# Assuming you have created temporary views 'Epilepsy_Cohort_Pair' and 'Epilepsy_Cohort_Pair1'

# SQL query to extract patients from 'Epilepsy_Cohort_Pair1' that are not present in 'Epilepsy_Cohort_Pair'
result = spark.sql("""
    SELECT table1.personid
    FROM Epilepsy_Cohort_Pair1 AS table1
    LEFT JOIN Epilepsy_Cohort_Pair AS table2 ON table1.personid = table2.personid
    WHERE table2.personid IS NULL
""")
# Count the number of patients in the result
count_of_patients = result.count()

# Show the count
print("Count of patients from table2 not present in table1:", count_of_patients)
# Show the result
result.show(truncate = False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+
|personid                            |
+------------------------------------+
|03554088-95b4-4c88-b907-d11790dadff3|
|0f16a7fe-6304-422d-92b1-76f6f90ae562|
|19a55932-df69-45af-9d4d-1bf946965e75|
|1cc1de4b-b8f4-4bc5-92f3-b8cf01cbdc00|
|1e5f9659-3542-498e-a182-55b22a5c6835|
|20770de0-9f56-41bb-b73c-3ca78fa33d9c|
|2093e723-a83f-45b9-8bcf-80218291910d|
|287d2da3-76aa-4cfd-9f99-1130dcce74f1|
|2effe481-a548-4c70-bfc4-6e68279c5668|
|338173d3-be22-4a9d-a167-76e288dce1bd|
|346d3b91-3126-419a-9e2b-bb51075a1a98|
|36b82274-eef5-4035-84ee-f151f77207c5|
|370abec9-e81f-4d65-a207-c08e92aa664b|
|3c42bb1c-caaa-46e5-8023-474ff1bf52ab|
|44cc2d70-7132-44bb-b30c-0234fee612a1|
|49cda762-229c-43e3-8db4-1323d8448ae8|
|4af7189c-d1b2-405d-bb60-df34fcb28bbe|
|4c45c4f0-fe60-481d-9737-79bc89485a71|
|4c5eb2b1-33d2-48d7-8f24-7cd44457b017|
|4f4908f8-afeb-400b-bf9f-ca8b6d6f0985|
+------------------------------------+
only showing top 20 rows



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
Count_Result = spark.sql("SELECT COUNT(DISTINCT personid) FROM Epilepsy_Cohort_Pair")
Count_Result.show(truncate = False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------+
|count(DISTINCT personid)|
+------------------------+
|140111                  |
+------------------------+



<IPython.core.display.Javascript object>

In [32]:
#Loading unmatched items for cohort
df = pd.read_csv("file:///home/o_suchsi/work/Oklahoma State/Priya/epilepsy/unmatched_items.csv")
# Define column headers as a list
column_headers = ['Unmatched Items']

# Assign column headers to the DataFrame
df.columns = column_headers
# # Display the first few rows of the DataFrame
# # print(df.head())
# display(df.head(10))
# print(len(df))
# Count the number of unique values in 'Column1'
unique_count = df['Unmatched Items'].nunique()
# Print the count of unique values
print("Number of Unique Values in 'DrugName':", unique_count)
unique_Med = df['Unmatched Items'].unique()
# Create a new DataFrame with unique values
unique_df = pd.DataFrame({'UniqueDrugName': unique_Med})

# Print the list of unique values
print("Unique Values in 'DrugName':")
print(unique_Med.tolist())
# display(unique_df)

▸,:,


Number of Unique Values in 'DrugName': 674
Unique Values in 'DrugName':
['ocular lubricant', 'Zinacef', 'Ancef', 'multivitamin', 'Toradol', 'physiological irrigating solution', 'Librium', 'Mucomyst', 'multivitamin with minerals', 'Culturelle DS', 'Rocephin', 'Sublimaze', 'Gastrografin', 'Versed', 'Flexeril', 'Vicks VapoSteam', 'Periactin', 'Lexiscan', 'Bicitra', 'phytonadione', 'OxyIR', 'Stadol', 'Norflex', 'Mylanta', 'Stuartnatal Plus', 'LET', 'Kefzol', 'Vicodin', 'Prenatal Multivitamins', 'Persantine IV', 'Normal Saline Flush', 'LR', 'multiple vitamins', 'Gadavist', 'Blistex Lip Balm', 'Isovue-370', 'Magnevist', 'Definity', 'Multiple Vitamins with Minerals', 'D5LR', 'Depacon', 'Carmex', 'Robitussin', 'Lactated Ringers', 'D5NS', 'Normodyne', 'Centrum', 'Lets Kit', 'Prenatabs Rx', 'Freestyle Lancet', 'Emla', 'Campral', 'Certa-Vite', 'Celebrate Multivitamin', 'D5W', 'Atarax', 'Kayexalate', 'Aredia', 'Nubain', 'Optiray 320', 'Multihance', 'Varibar Thin', 'Halfprin', 'Dotarem', 'Capoten',

In [6]:
#Final List from cohort and control med combined list
df = pd.read_csv("file:///home/o_suchsi/work/Oklahoma State/Priya/epilepsy/unmatched_items_latest.csv")
# Define column headers as a list
column_headers = ['Unmatched Items']

# Assign column headers to the DataFrame
df.columns = column_headers
# # Display the first few rows of the DataFrame
# # print(df.head())
# display(df.head(10))
# print(len(df))
# Count the number of unique values in 'Column1'
unique_count = df['Unmatched Items'].nunique()
# Print the count of unique values
print("Number of Unique Values in 'DrugName':", unique_count)
unique_Med_final = df['Unmatched Items'].unique()
# # Create a new DataFrame with unique values
# unique_df = pd.DataFrame({'UniqueDrugName': unique_Med})

# Print the list of unique values
print("Unique Values in 'DrugName':")
print(unique_Med_final.tolist())
# display(unique_df)

▸,:,


Number of Unique Values in 'DrugName': 1008
Unique Values in 'DrugName':
['ocular lubricant', 'Zinacef', 'Ancef', 'multivitamin', 'Toradol', 'physiological irrigating solution', 'Librium', 'Mucomyst', 'multivitamin with minerals', 'Culturelle DS', 'Rocephin', 'Sublimaze', 'Gastrografin', 'Versed', 'Flexeril', 'Vicks VapoSteam', 'Periactin', 'Lexiscan', 'Bicitra', 'phytonadione', 'OxyIR', 'Stadol', 'Norflex', 'Mylanta', 'Stuartnatal Plus', 'LET', 'Kefzol', 'Vicodin', 'Prenatal Multivitamins', 'Persantine IV', 'Normal Saline Flush', 'LR', 'multiple vitamins', 'Gadavist', 'Blistex Lip Balm', 'Isovue-370', 'Magnevist', 'Definity', 'Multiple Vitamins with Minerals', 'D5LR', 'Depacon', 'Carmex', 'Robitussin', 'Lactated Ringers', 'D5NS', 'Normodyne', 'Centrum', 'Lets Kit', 'Prenatabs Rx', 'emollients', 'Freestyle Lancet', 'Emla', 'Campral', 'Certa-Vite', 'Celebrate Multivitamin', 'D5W', 'Atarax', 'Kayexalate', 'Aredia', 'Nubain', 'Optiray 320', 'Multihance', 'Varibar Thin', 'Halfprin', 'Dotar

In [28]:
#unmatched items loaded for control group
df = pd.read_csv("file:///home/o_suchsi/work/Oklahoma State/Priya/epilepsy/unmatched_items_control_latest.csv")
# Define column headers as a list
column_headers = ['Unmatched Items']

# Assign column headers to the DataFrame
df.columns = column_headers
# # Display the first few rows of the DataFrame
# # print(df.head())
# display(df.head(10))
# print(len(df))
# Count the number of unique values in 'Column1'
unique_count = df['Unmatched Items'].nunique()
# Print the count of unique values
print("Number of Unique Values in 'DrugName':", unique_count)
unique_Med_Control = df['Unmatched Items'].unique()
# # Create a new DataFrame with unique values
# unique_df = pd.DataFrame({'UniqueDrugName': unique_Med})

# Print the list of unique values
print("Unique Values in 'DrugName':")
print(unique_Med_Control.tolist())
# display(unique_df)

▸,:,


Number of Unique Values in 'DrugName': 868
Unique Values in 'DrugName':
['drugname', 'Rocephin', 'multivitamin with minerals', 'Toradol', 'multiple vitamins', 'Norflex', 'Gastrografin', 'Gadavist', 'MD-Gastroview', 'Optiray 320', 'Flexeril', 'Robitussin', 'multivitamin', 'phytonadione', 'Lets Kit', 'Lumason', 'Isovue-370', 'Normal Saline Flush', 'Prenatal Multivitamins', 'Carmex', 'Definity', 'StressTabs', 'Mylanta', 'Bicitra', 'Ancef', 'Unknown', 'PreNata', 'D10W', 'AeroChamber Plus Flow-Vu', 'ocular lubricant', 'Imdur', 'HPV', 'LR', 'Mag64', 'Robitussin-AC', 'Foltanx', 'Levothroid', 'Tdap', 'D5LR', 'Versed', 'D5NS', 'Nephro-Vite', 'Librium', 'Amoxil', 'Thorazine', 'Fluvirin', 'Kayexalate', 'Flintstones Multivitamins', 'Wheelchair', 'Dotarem', 'Sublimaze', 'MMR', 'Prenatabs Rx', 'Varibar Thin', 'Varibar Pudding', 'Varibar Nectar', 'Maltsupex', 'Mononessa', 'Celebrate Multivitamin', 'OxyIR', 'Lexiscan', 'Neosporin', 'Multiple Vitamins with Minerals', 'Nubain', 'Multiple Vitamins', 'Rom

In [43]:
#Finalized code to count the no of patients in cohort having more than 1% of those med consumed
from pyspark.sql.functions import col, count, sum

# # Assuming you have a DataFrame named 'Epilepsy_Cohort_Paired_New' with a 'drugname' column
# # and a list named 'unique_values_list' with unique drug names to compare

# # Step 1: Calculate the total number of patients
# total_patients = Epilepsy_Cohort_Paired_New.select("personid").distinct().count()

# # Step 2: Create a DataFrame to count patients for each unique drug name
# drugname_counts = Epilepsy_Cohort_Paired_New.groupBy("drugname").agg(
#     (count("*") / total_patients * 100).alias("percentage_of_total_patients")
# )

# # Step 3: Filter the DataFrame to include only drug names with counts > 1% of total patients
# filtered_drugnames = drugname_counts.filter(col("percentage_of_total_patients") > 1)

from pyspark.sql.functions import col, count, lit

from pyspark.sql.types import StringType

# Convert the Python list 'unique_Med' into a list of literals
unique_Med_literals = [lit(value) for value in unique_Med]

# Use 'unique_Med_literals' in the 'isin' method to filter the DataFrame
filtered_df = Epilepsy_Cohort_Paired_New.filter(col("drugname").isin(unique_Med_literals))

# Print the filtered DataFrame
print(filtered_df.count())

# Step 1: Calculate the total number of patients
total_patients = Epilepsy_Cohort_Paired_New.select("personid").distinct().count()

# Step 2: Create a DataFrame to count patients for each unique drug name
drugname_counts = filtered_df.groupBy("drugname").agg(
    count("*").alias("patient_count"),
    (count("*") / total_patients * 100).alias("percentage_of_total_patients")
)

# Step 3: Filter the DataFrame to include only drug names with counts > 1% of total patients
filtered_drugnames = drugname_counts.filter(col("percentage_of_total_patients") > 1)

# Show or use the resulting DataFrame 'filtered_drugnames'
filtered_drugnames.show()
# Step 4: Count the total number of such drug names
total_drugnames_more_than_1_percent = filtered_drugnames.count()

# Show the count
print("Total drug names with more than 1% of total patients:", total_drugnames_more_than_1_percent)

# Calculate the total of 'patient_count' column
total_patient_count = filtered_drugnames.agg(sum("patient_count")).collect()[0][0]

# Show the total patient count
print("Total patient count:", total_patient_count)

# You can count the distinct 'personid' values to get the total count
total_personid_count = Epilepsy_Cohort_Paired_New.select("personid").distinct().count()

# Show the total count
print("Total number of unique personid values:", total_personid_count)
# unique_values_list = filtered_drugnames['drugname'].unique()
# for i in unique_values_list:
#     print(i)
# Extract unique drug names from the 'filtered_drugnames' DataFrame
unique_drugnames = filtered_drugnames.select("drugname").distinct()
# Extract unique drug names from the 'filtered_drugnames' DataFrame
unique_drugnames_count = filtered_drugnames.select("drugname").distinct().count()
print("drug count", unique_drugnames_count)
# # Show the unique drug names
# unique_drugnames.show(342, truncate=False)
# Extract the column from the DataFrame and collect it into a list
column_values_list = unique_drugnames.select("drugname").collect()

# Convert the collected values to a Python list
column_values = [row.drugname for row in column_values_list]

for i in column_values:
    print(i)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

74208


<IPython.core.display.Javascript object>

...

drug count 10


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Ancef
Versed
LR
Toradol
Rocephin
ocular lubricant
multivitamin with minerals
Flexeril
multivitamin
Normal Saline Flush


<IPython.core.display.Javascript object>

In [29]:
#Finalized code to count the no of patients in control having more than 1% of those med consumed
from pyspark.sql.functions import col, count, sum

# # Assuming you have a DataFrame named 'Epilepsy_Cohort_Paired_New' with a 'drugname' column
# # and a list named 'unique_values_list' with unique drug names to compare

# # Step 1: Calculate the total number of patients
# total_patients = Epilepsy_Cohort_Paired_New.select("personid").distinct().count()

# # Step 2: Create a DataFrame to count patients for each unique drug name
# drugname_counts = Epilepsy_Cohort_Paired_New.groupBy("drugname").agg(
#     (count("*") / total_patients * 100).alias("percentage_of_total_patients")
# )

# # Step 3: Filter the DataFrame to include only drug names with counts > 1% of total patients
# filtered_drugnames = drugname_counts.filter(col("percentage_of_total_patients") > 1)

from pyspark.sql.functions import col, count, lit

from pyspark.sql.types import StringType

# Convert the Python list 'unique_Med' into a list of literals
unique_Med_literals = [lit(value) for value in unique_Med_Control]

# Use 'unique_Med_literals' in the 'isin' method to filter the DataFrame
filtered_df = Epilepsy_Control_Paired_New.filter(col("drugname").isin(unique_Med_literals))

# Print the filtered DataFrame
print(filtered_df.count())

# Step 1: Calculate the total number of patients
total_patients = Epilepsy_Control_Paired_New.select("personid").distinct().count()

# Step 2: Create a DataFrame to count patients for each unique drug name
drugname_counts = filtered_df.groupBy("drugname").agg(
    count("*").alias("patient_count"),
    (count("*") / total_patients * 100).alias("percentage_of_total_patients")
)

# Step 3: Filter the DataFrame to include only drug names with counts > 1% of total patients
filtered_drugnames = drugname_counts.filter(col("percentage_of_total_patients") > 1)

# Show or use the resulting DataFrame 'filtered_drugnames'
filtered_drugnames.show()
# Step 4: Count the total number of such drug names
total_drugnames_more_than_1_percent = filtered_drugnames.count()

# Show the count
print("Total drug names with more than 1% of total patients:", total_drugnames_more_than_1_percent)

# Calculate the total of 'patient_count' column
total_patient_count = filtered_drugnames.agg(sum("patient_count")).collect()[0][0]

# Show the total patient count
print("Total patient count:", total_patient_count)

# You can count the distinct 'personid' values to get the total count
total_personid_count = Epilepsy_Control_Paired_New.select("personid").distinct().count()

# Show the total count
print("Total number of unique personid values:", total_personid_count)
# unique_values_list = filtered_drugnames['drugname'].unique()
# for i in unique_values_list:
#     print(i)
# Extract unique drug names from the 'filtered_drugnames' DataFrame
unique_drugnames = filtered_drugnames.select("drugname").distinct()
# Extract unique drug names from the 'filtered_drugnames' DataFrame
unique_drugnames_count = filtered_drugnames.select("drugname").distinct().count()
print("drug count", unique_drugnames_count)
# # Show the unique drug names
# unique_drugnames.show(342, truncate=False)
# Extract the column from the DataFrame and collect it into a list
column_values_list = unique_drugnames.select("drugname").collect()

# Convert the collected values to a Python list
column_values = [row.drugname for row in column_values_list]

for i in column_values:
    print(i)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

131490


<IPython.core.display.Javascript object>

...

drug count 5


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

LR
Toradol
Rocephin
multivitamin
Normal Saline Flush


<IPython.core.display.Javascript object>

In [33]:
#Finalized code to combine both dataframes
import pandas as pd

# # Assuming you have the dataframes Epilepsy_Cohort_Paired_New and Epilepsy_Control_Paired_New

# # Extract unique records from the first dataframe
# unique_df1 = Epilepsy_Cohort_Paired_New.drop_duplicates(subset=['personid', 'drugname'])

# # Extract unique records from the second dataframe
# unique_df2 = Epilepsy_Control_Paired_New.drop_duplicates(subset=['personid', 'drugname'])

# # Concatenate the two dataframes to create a single dataframe
# combined_df = pd.concat([unique_df1, unique_df2])

# # Reset the index of the combined dataframe
# combined_df.reset_index(drop=True, inplace=True)
# Use union or unionAll to concatenate Spark DataFrames
combined_df = Epilepsy_Cohort_Paired_New.union(Epilepsy_Control_Paired_New)  # To remove duplicates

# You can optionally drop duplicates if needed
combined_df = combined_df.dropDuplicates()

# Now, combined_df contains the unique records from both dataframes
print(combined_df)

▸,:,


DataFrame[personid: string, drugname: string]


In [36]:
#Finalized code to count the no of patients in both combined having more than 1% of those med consumed
from pyspark.sql.functions import col, count, sum

# # Assuming you have a DataFrame named 'Epilepsy_Cohort_Paired_New' with a 'drugname' column
# # and a list named 'unique_values_list' with unique drug names to compare

# # Step 1: Calculate the total number of patients
# total_patients = Epilepsy_Cohort_Paired_New.select("personid").distinct().count()

# # Step 2: Create a DataFrame to count patients for each unique drug name
# drugname_counts = Epilepsy_Cohort_Paired_New.groupBy("drugname").agg(
#     (count("*") / total_patients * 100).alias("percentage_of_total_patients")
# )

# # Step 3: Filter the DataFrame to include only drug names with counts > 1% of total patients
# filtered_drugnames = drugname_counts.filter(col("percentage_of_total_patients") > 1)

from pyspark.sql.functions import col, count, lit

from pyspark.sql.types import StringType

# Convert the Python list 'unique_Med' into a list of literals
unique_Med_literals = [lit(value) for value in unique_Med_final]

# Use 'unique_Med_literals' in the 'isin' method to filter the DataFrame
filtered_df = combined_df.filter(col("drugname").isin(unique_Med_literals))

# Print the filtered DataFrame
print(filtered_df.count())

# Step 1: Calculate the total number of patients
total_patients = combined_df.select("personid").distinct().count()

# Step 2: Create a DataFrame to count patients for each unique drug name
drugname_counts = filtered_df.groupBy("drugname").agg(
    count("*").alias("patient_count"),
    (count("*") / total_patients * 100).alias("percentage_of_total_patients")
)

# Step 3: Filter the DataFrame to include only drug names with counts > 1% of total patients
filtered_drugnames = drugname_counts.filter(col("percentage_of_total_patients") > 1)

# Show or use the resulting DataFrame 'filtered_drugnames'
filtered_drugnames.show()
# Step 4: Count the total number of such drug names
total_drugnames_more_than_1_percent = filtered_drugnames.count()

# Show the count
print("Total drug names with more than 1% of total patients:", total_drugnames_more_than_1_percent)

# Calculate the total of 'patient_count' column
total_patient_count = filtered_drugnames.agg(sum("patient_count")).collect()[0][0]

# Show the total patient count
print("Total patient count:", total_patient_count)

# You can count the distinct 'personid' values to get the total count
total_personid_count = combined_df.select("personid").distinct().count()

# Show the total count
print("Total number of unique personid values:", total_personid_count)
# unique_values_list = filtered_drugnames['drugname'].unique()
# for i in unique_values_list:
#     print(i)
# Extract unique drug names from the 'filtered_drugnames' DataFrame
unique_drugnames = filtered_drugnames.select("drugname").distinct()
# Extract unique drug names from the 'filtered_drugnames' DataFrame
unique_drugnames_count = filtered_drugnames.select("drugname").distinct().count()
print("drug count", unique_drugnames_count)
# # Show the unique drug names
# unique_drugnames.show(342, truncate=False)
# Extract the column from the DataFrame and collect it into a list
column_values_list = unique_drugnames.select("drugname").collect()

# Convert the collected values to a Python list
column_values = [row.drugname for row in column_values_list]

for i in column_values:
    print(i)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Ancef
LR
Toradol
Rocephin
multivitamin
Normal Saline Flush


<IPython.core.display.Javascript object>

In [ ]:
from pyspark.sql.functions import col, count, sum

# # Assuming you have a DataFrame named 'Epilepsy_Cohort_Paired_New' with a 'drugname' column
# # and a list named 'unique_values_list' with unique drug names to compare

# # Step 1: Calculate the total number of patients
# total_patients = Epilepsy_Cohort_Paired_New.select("personid").distinct().count()

# # Step 2: Create a DataFrame to count patients for each unique drug name
# drugname_counts = Epilepsy_Cohort_Paired_New.groupBy("drugname").agg(
#     (count("*") / total_patients * 100).alias("percentage_of_total_patients")
# )

# # Step 3: Filter the DataFrame to include only drug names with counts > 1% of total patients
# filtered_drugnames = drugname_counts.filter(col("percentage_of_total_patients") > 1)

from pyspark.sql.functions import col, count, lit

from pyspark.sql.types import StringType

# Convert the Python list 'unique_Med' into a list of literals
unique_Med_literals = [lit(value) for value in unique_Med]

# Use 'unique_Med_literals' in the 'isin' method to filter the DataFrame
filtered_df = Epilepsy_Cohort_Paired_New.filter(col("drugname").isin(unique_Med_literals))

# Print the filtered DataFrame
print(filtered_df.count())

# Step 1: Calculate the total number of patients
total_patients = Epilepsy_Cohort_Paired_New.select("personid").distinct().count()

# Step 2: Create a DataFrame to count patients for each unique drug name
drugname_counts = filtered_df.groupBy("drugname").agg(
    count("*").alias("patient_count"),
    (count("*") / total_patients * 100).alias("percentage_of_total_patients")
)

# Step 3: Filter the DataFrame to include only drug names with counts > 1% of total patients
filtered_drugnames = drugname_counts.filter(col("percentage_of_total_patients") > 1)

# Show or use the resulting DataFrame 'filtered_drugnames'
filtered_drugnames.show()
# Step 4: Count the total number of such drug names
total_drugnames_more_than_1_percent = filtered_drugnames.count()

# Show the count
print("Total drug names with more than 1% of total patients:", total_drugnames_more_than_1_percent)

# Calculate the total of 'patient_count' column
total_patient_count = filtered_drugnames.agg(sum("patient_count")).collect()[0][0]

# Show the total patient count
print("Total patient count:", total_patient_count)

# You can count the distinct 'personid' values to get the total count
total_personid_count = Epilepsy_Cohort_Paired_New.select("personid").distinct().count()

# Show the total count
print("Total number of unique personid values:", total_personid_count)
# unique_values_list = filtered_drugnames['drugname'].unique()
# for i in unique_values_list:
#     print(i)
# Extract unique drug names from the 'filtered_drugnames' DataFrame
unique_drugnames = filtered_drugnames.select("drugname").distinct()
# Extract unique drug names from the 'filtered_drugnames' DataFrame
unique_drugnames_count = filtered_drugnames.select("drugname").distinct().count()
print("drug count", unique_drugnames_count)
# # Show the unique drug names
# unique_drugnames.show(342, truncate=False)
# Extract the column from the DataFrame and collect it into a list
column_values_list = unique_drugnames.select("drugname").collect()

# Convert the collected values to a Python list
column_values = [row.drugname for row in column_values_list]

for i in column_values:
    print(i)

In [51]:
# Step 3: Filter the DataFrame to include only drug names with counts > 1% of total patients - cohort
filtered_drugnames1 = drugname_counts.filter(col("percentage_of_total_patients") <= 1)
# Show or use the resulting DataFrame 'filtered_drugnames'
filtered_drugnames1.show()
# Step 4: Count the total number of such drug names
total_drugnames_less_than_or_equal_to_1_percent = filtered_drugnames1.count()

# Show the count
print("Total drug names with <= 1% of total patients:", total_drugnames_less_than_or_equal_to_1_percent)

# Calculate the total of 'patient_count' column
total_patient_count = filtered_drugnames1.agg(sum("patient_count")).collect()[0][0]

# Show the total patient count
print("Total patient count:", total_patient_count)

# You can count the distinct 'personid' values to get the total count
total_personid_count = Epilepsy_Cohort_Paired_New.select("personid").distinct().count()

# Show the total count
print("Total number of unique personid values:", total_personid_count)
# unique_values_list = filtered_drugnames['drugname'].unique()
# for i in unique_values_list:
#     print(i)
# Extract unique drug names from the 'filtered_drugnames' DataFrame
unique_drugnames = filtered_drugnames1.select("drugname").distinct()
# Extract unique drug names from the 'filtered_drugnames' DataFrame
unique_drugnames_count = filtered_drugnames1.select("drugname").distinct().count()
print("drug count", unique_drugnames_count)
# # Show the unique drug names
# unique_drugnames.show(342, truncate=False)
# Extract the column from the DataFrame and collect it into a list
column_values_list = unique_drugnames.select("drugname").collect()

# # Convert the collected values to a Python list
# column_values = [row.drugname for row in column_values_list]

# for i in column_values:
#     print(i)
# Convert the collected values to a Python list
column_values = [row.drugname for row in column_values_list]
for i in column_values:
    print(i)

# # Check if 'cholecalciferol' is in the list
# if 'cholecalciferol' in column_values:
#     print("'cholecalciferol' is in the list")
# else:
#     print("'cholecalciferol' is not in the list")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+--------------------+-------------+----------------------------+
|            drugname|patient_count|percentage_of_total_patients|
+--------------------+-------------+----------------------------+
|             Volumen|            8|        0.005709758691323308|
|ACCU-CHECK Guide ...|            3|        0.002141159509246...|
|             Lumason|           46|        0.032831112475109024|
|             Biotene|            7|        0.004996038854907895|
|          pycnogenol|            1|        7.137198364154135E-4|
|CertaVite with An...|            3|        0.002141159509246...|
|            SINEquan|            5|        0.003568599182077...|
|            Gadavist|         1115|          0.7957976176031861|
|              Deplin|            8|        0.005709758691323308|
|             Dolobid|            4|        0.002854879345661654|
|     Nyquil Liquicap|            2|        0.001427439672830827|
|    CertaVite Senior|            4|        0.002854879345661654|
|         

<IPython.core.display.Javascript object>

...

drug count 664


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Volumen
ACCU-CHECK Guide Meter
Lumason
Biotene
pycnogenol
CertaVite with Antioxidants
SINEquan
Gadavist
Deplin
Dolobid
Nyquil Liquicap
CertaVite Senior
Renaphro
Bengay Pain Relief+Massage
Apatate Forte
Pletal
MacuGuard
Icy Hot Extra Strength
LET
NeilMed Sinus Rinse Kit
ICaps AREDS
Luminal
Phenazo
Refresh Optive
Easprin
Aveeno Soothing Bath Treatment
Robitussin CF
Nifedical XL
Entero-H
Bismatrol
Pavulon
MSM
Vortex with Mask Toddler
Softclix Lancets
Expecta DHA
Eloxatin
Lancing Device
Septra
Optison
ADEKs
NovaFerrum Pediatric
Prednisol
Definity
Prolixin
Aurodex
Flintstones Multivitamins
Cervarix
Nohist-DMX
Nasex
NTG
Cerefolin
Geritol Complete
Accu-Chek Connect Meter
PNV Prenatal
Gingko Biloba
Accu-Chek Compact Test Strips
Albolene Moisturizing Cleanser
Auralgan
Prenate AM
Lancet
Prenatal DHA
Ben Gay
Ascriptin Enteric
LVP solution with hypertonic saline
Prenate Elite
Prenate DHA
Flanax Pain Reliever
LVP solution
AeroChamber Plus Flow-Vu
IPV
Prosom
Omnicef
D20W
NovoFine 32g Needle
Mouth Ko

<IPython.core.display.Javascript object>

In [52]:
# Assuming you have created temporary views 'Epilepsy_Cohort_Pair' and 'Epilepsy_Cohort_Pair1'

# SQL query to extract patients from 'Epilepsy_Cohort_Pair1' that are not present in 'Epilepsy_Cohort_Pair'
result = spark.sql("""
    SELECT DISTINCT count( personid)
    FROM Epilepsy_Cohort_Pair where drugname = 'multiple vitamins'
""")
# # Count the number of patients in the result
# count_of_patients = result.count()

# # Show the count
# print("Count of patients from table2 not present in table1:", count_of_patients)
# Show the result
result.show(truncate = False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+---------------+
|count(personid)|
+---------------+
|393            |
+---------------+



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Print the first row of the drugname column
Epilepsy_Cohort_Paired_New.select("drugname").show(100, truncate=False)

In [18]:
import csv
# Select the 'drugname' column from the DataFrame
drugname_column = Epilepsy_Control_Paired_New.select("drugname")

# Collect the values from the DataFrame into a list
drugname_values = [row.drugname for row in drugname_column.collect()]

# Specify the path for the output CSV file
output_csv_file_path = "/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/drugname_values_control_latest.csv"

# Create a CSV file and write the values
with open(output_csv_file_path, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    
    # Write a header if needed
    writer.writerow(["drugname"])  # Uncomment this line to include a header
    
    # Write the values from the drugname_values list
    for value in drugname_values:
        writer.writerow([value])

print("CSV file saved at:", output_csv_file_path)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

CSV file saved at: /home/o_suchsi/work/Oklahoma State/Priya/epilepsy/drugname_values_control_latest.csv


In [60]:
!hadoop fs -copyToLocal drugname_values1.txt ~/work/Oklahoma%20State/Priya/epilepsy

▸,:,


copyToLocal: `drugname_values1.txt': No such file or directory


In [20]:
# Read the contents of the second file into another Pandas DataFrame
df3 = pd.read_csv('drugname_values_control_latest.csv', header=None, names=['drugname'])
# Extract unique items into a new DataFrame
unique_df_control = df3.drop_duplicates()
# Extract the 'drugname' column as a Pandas Series
unique_drugname_series = unique_df_control['drugname']

# Convert the Pandas Series to a list
unique_drugname_list_control = unique_drugname_series.tolist()

▸,:,


In [26]:
#Run this!!!-Finalized Normalization code for Medication with control alone
import re
# import docx
import csv
# Initialize counts for matched and unmatched items in unique_values_list
total_matched_item_count_list = 0
total_unmatched_item_count_list = 0
total_matched_items_list = []
total_unmatched_items_list = []

# Function to check if a word is numeric
def is_numeric(word):
    try:
        float(word)
        return True
    except ValueError:
        return False

# Iterate over elements in unique_values_list
for val1 in unique_drugname_list_control:
    # Split the words in val1 using space or special characters
    words1 = re.split(r'[ \W]', val1)
    # Take the first word from words1 (lowercase)
    first_word1 = words1[0].lower() if words1 else ""
    # print(first_word1)

    # Initialize a flag for matching against unique_values_list1
    matched1 = False

    # Iterate over elements in unique_values_list1
    for val2 in unique_values_list1:
        # Split the words in val2 using space or special characters
        words2 = re.split(r'[ \W]', val2)
        # print(words2)

        # Check if the first word from val1 (lowercase) matches any word from val2 (lowercase)
        if first_word1 in [word.lower() for word in words2]:
            matched1 = True
            break  # No need to check further for this element in unique_values_list1

    # Initialize a flag for matching against unique_values_list1
    matched2 = False

    # Iterate over elements in unique_values_list1
    for val2 in unique_values_list1:
        # Split the words in val2 using space or special characters
        words2 = re.split(r'[ \W]', val2)
        # Take the first word from words2 (lowercase)
        first_word2 = words2[0].lower() if words2 else ""
        # print(words2)

        # Check if the first word from val2 (lowercase) matches any word from val1 (lowercase)
        if first_word2 in [word.lower() for word in words1]:
            matched2 = True
            break  # No need to check further for this element in unique_values_list1

    # Increment counts and add val1 to lists based on match status
    if matched1 or matched2:
        total_matched_item_count_list += 1
        total_matched_items_list.append(val1)
    else:
        total_unmatched_item_count_list += 1
        total_unmatched_items_list.append(val1)

# Print the counts and values from unique_values_list
print("Total count of matched items in unique_values_list:", total_matched_item_count_list)
print("Total count of unmatched items in unique_values_list:", total_unmatched_item_count_list)
# print("Matched items in unique_values_list:", total_matched_items_list)
# print("Unmatched items in unique_values_list:", total_unmatched_items_list)

# # Create DataFrames for matched and unmatched items
# matched_items_df = pd.DataFrame({'matched_items': total_matched_items_list})
# unmatched_items_df = pd.DataFrame({'unmatched_items': total_unmatched_items_list})

# # Specify the Excel file paths where you want to save the data
# matched_excel_file_path = 'matched_output.xlsx'
# unmatched_excel_file_path = 'unmatched_output.xlsx'

# # Write the DataFrames to separate Excel files
# matched_items_df.to_excel(matched_excel_file_path, index=False)
# unmatched_items_df.to_excel(unmatched_excel_file_path, index=False)
# Save matched and unmatched items to a CSV file
with open('matched_items_control_latest.csv', 'w', newline='') as csvfile:
    fieldnames = ['Matched Items']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

    writer.writeheader()
    for matched_item in total_matched_items_list:
        writer.writerow({'Matched Items': matched_item})

print("CSV file 'matched_items_control_latest.csv' has been created with matched and unmatched items.")
with open('unmatched_items_control_latest.csv', 'w', newline='') as csvfile:
    fieldnames = ['Unmatched Items']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

    writer.writeheader()
    for unmatched_item in total_unmatched_items_list:
        writer.writerow({'Unmatched Items': unmatched_item})

print("CSV file 'unmatched_items_control_latest.csv' has been created with unmatched items.")

▸,:,


Total count of matched items in unique_values_list: 28632
Total count of unmatched items in unique_values_list: 868
CSV file 'matched_items_control_latest.csv' has been created with matched and unmatched items.
CSV file 'unmatched_items_control_latest.csv' has been created with unmatched items.


In [11]:
import pandas as pd

# Read the contents of the first file into a Pandas DataFrame
df1 = pd.read_csv('drugname_values.txt', header=None, names=['drugname'])

# Read the contents of the second file into another Pandas DataFrame
df2 = pd.read_csv('drugname_values_control_latest.csv', header=None, names=['drugname'])

# Concatenate the two DataFrames to combine them
combined_df = pd.concat([df1, df2])

# Extract unique items into a new DataFrame
unique_df = combined_df.drop_duplicates()

# Count the number of items in the unique DataFrame
count = unique_df.shape[0]
print("Number of unique items:", count)
# Extract the 'drugname' column as a Pandas Series
unique_drugname_series = unique_df['drugname']

# Convert the Pandas Series to a list
unique_drugname_list_combined = unique_drugname_series.tolist()

# # Print the unique items as a list
# print(unique_drugname_list)

# # Print the unique items
# for item in unique_df['drugname']:
#     print(item)

▸,:,


Number of unique items: 33914


In [ ]:
# # Select the 'drugname' column from the DataFrame
# drugname_column = Epilepsy_Cohort_Paired_New.select("drugname")

# # Collect the values from the DataFrame into a list
# drugname_values = [row.drugname for row in drugname_column.collect()]

# # Specify the path for the output .txt file
# output_text_file_path = "file:///home/o_suchsi/work/Oklahoma State/Priya/epilepsy/drugname_values.txt"

# # # Convert the list of values to a DataFrame with a single column
# # from pyspark.sql import SparkSession
# # spark = SparkSession.builder.getOrCreate()
# values_df = spark.createDataFrame(drugname_values, ["value"])

# # Write the values to the .txt file using write.mode("overwrite").text()
# values_df.write.mode("overwrite").text(output_text_file_path)

# Select the 'drugname' column from the DataFrame
drugname_column = Epilepsy_Cohort_Paired_New.select("drugname")

# Collect the values from the DataFrame into a list
drugname_values = [row.drugname for row in drugname_column.collect()]

# Specify the path for the output .txt file
output_text_file_path = "drugname_values.txt"

# Write the values to the .txt file
with open(output_text_file_path, "w+") as txt_file:
    for value in drugname_values:
        txt_file.write(value + "\n")

In [5]:
df = pd.read_csv("file:///home/o_suchsi/work/Oklahoma State/Priya/epilepsy/drugname_values.txt", sep='\t')
# Define column headers as a list
column_headers = ['DrugName']

# Assign column headers to the DataFrame
df.columns = column_headers
# # Display the first few rows of the DataFrame
# # print(df.head())
# display(df.head(10))
# print(len(df))
# Count the number of unique values in 'Column1'
unique_count = df['DrugName'].nunique()
# Print the count of unique values
print("Number of Unique Values in 'DrugName':", unique_count)
unique_values_list = df['DrugName'].unique()
# Create a new DataFrame with unique values
unique_df = pd.DataFrame({'UniqueDrugName': unique_values_list})

# Print the list of unique values
print("Unique Values in 'DrugName':")
print(unique_values_list.tolist())
# display(unique_df)

▸,:,


Number of Unique Values in 'DrugName': 22629
Unique Values in 'DrugName':
['vecuronium', 'iohexol', 'citric acid-potassium bicarbonate', 'rocuronium', 'vancomycin', 'heparin', 'fentaNYL', 'propofol', 'EPINEPHrine', 'lidocaine', 'heparin flush', 'bacitracin-polymyxin B topical', 'dexamethasone', 'iodixanol', 'atropine', 'meropenem', 'metoclopramide', 'ondansetron', 'ampicillin-sulbactam', 'indomethacin', 'insulin lispro', 'ceFAZolin', 'thrombin topical', 'HYDROmorphone', 'piperacillin-tazobactam', 'epinephrine-lidocaine', 'potassium chloride', 'succinylcholine', 'LORazepam', 'Sodium Chloride 3% intravenous solution', 'benztropine', 'spironolactone', 'morphine', 'aztreonam', 'sodium chloride', 'sugammadex', 'ocular lubricant', 'etomidate', 'Zoloft', 'ibuprofen', 'cefTRIAXone', 'Zofran', 'isosorbide mononitrate', 'thiamine 100 mg oral tablet', 'niacin', 'Zinacef', 'Haldol Decanoate 100 mg/mL intramuscular solution', 'Wellbutrin SR', 'Sodium Chloride 0.9% intravenous solution', 'Ancef', 'a

In [9]:
# df1 = pd.read_csv("file:///home/o_suchsi/work/Oklahoma State/Priya/epilepsy/pharmacy_samps.csv")
df1 = pd.read_csv("file:///home/o_suchsi/work/Oklahoma State/Priya/epilepsy/RXNORM.csv")
# # Display the first few rows of the DataFrame
# print(df1.head())
# display(df1.head(10))
# Print the column headers
column_headers = df1.columns
# for header in column_headers:
#     print(header)
# print(len(df1))
# Count the number of unique values in 'Column1'
unique_count = df1['Preferred Label'].nunique()
# Print the count of unique values
# print("Number of Unique Values in Preferred Label:", unique_count)
unique_values_list1 = df1['Preferred Label'].unique()
# Print the list of unique values
unique_values_list1.tolist()
# print("Unique Values in 'Preferred Label':")
# print(unique_values_list1.tolist())

▸,:,


['fentanyl 0.3 MG Buccal Tablet',
 'strawberry juice',
 'chlorthalidone 15 MG Oral Tablet [Thalitone]',
 'glimepiride 4 MG / rosiglitazone 8 MG Oral Tablet',
 'sodium acrylate/sodium acryloyldimethyltaurate copolymer (4000000 MW)',
 'Vectibix Injectable Product',
 'white ash pollen extract 1 MG/ML Injectable Solution',
 'Abuse-Deterrent oxycodone hydrochloride 5 MG Oral Tablet [Roxybond]',
 'Scutellaria barbata whole extract',
 'chlorthalidone 25 MG Oral Tablet [Thalitone]',
 'loteprednol etabonate Ophthalmic Suspension [Lotemax]',
 'trimethylamine',
 'menthol 0.027 MG/MG',
 'omeprazole 0.37 MG/MG Oral Paste [UlcerGard]',
 'Belsomra',
 'OsCal 500',
 'Injectable Foam',
 'bupivacaine Injectable Product',
 'aluminum hydroxide 600 MG',
 'zinc sulfate monohydrate',
 'methadone hydrochloride 5 MG/ML',
 'glycolate Topical Lotion',
 '1,1,3-tri(3-tert-butyl-4-hydroxy-6-methylphenyl)butane',
 'pegfilgrastim Prefilled Syringe [Udenyca]',
 'deferasirox 250 MG Tablet for Oral Suspension',
 'levobet

In [8]:
import docx
import re
# Initialize counts for matched and unmatched items in unique_values_list
matched_item_count_list = 0
unmatched_item_count_list = 0
matched_items_list = []
unmatched_items_list = []

# Function to check if a word is numeric
def is_numeric(word):
    try:
        float(word)
        return True
    except ValueError:
        return False

# Iterate over elements in unique_values_list
for val1 in unique_values_list:
    # Split the words in val1 using space or special characters
    words1 = re.split(r'[ \W]', val1)
    # Take the first word from words1 (lowercase)
    first_word1 = words1[0].lower() if words1 else ""
    # print(first_word1)
    matched = False  # Flag to check if any match is found

    # Iterate over elements in unique_values_list1
    for val2 in unique_values_list1:
        # Split the words in val2 using space or special characters
        words2 = re.split(r'[ \W]', val2)
        # print(words2)

        # Check if the first word from val1 (lowercase) matches any word from val2 (lowercase)
        if first_word1 in [word.lower() for word in words2]:
            matched = True
            break  # No need to check further for this element in unique_values_list1

    # Increment counts based on match status and add items to lists
    if matched:
        matched_item_count_list += 1
        matched_items_list.append(val1)
    else:
        unmatched_item_count_list += 1
        unmatched_items_list.append(val1)

# Reset counts for matched and unmatched items
matched_item_count_list2 = 0
unmatched_item_count_list2 = 0
matched_items_list2 = []
unmatched_items_list2 = []

# Iterate over elements in unique_values_list1 and compare them to unique_values_list
for val2 in unique_values_list1:
    # Split the words in val2 using space or special characters
    words2 = re.split(r'[ \W]', val2)
    # Take the first word from words2 (lowercase)
    # print(words2)
    first_word2 = words2[0].lower() if words2 else ""
    matched = False  # Flag to check if any match is found

    # Iterate over elements in unique_values_list
    for val1 in unique_values_list:
        # Split the words in val1 using space or special characters
        words1 = re.split(r'[ \W]', val1)

        # Check if the first word from val2 (lowercase) matches any word from val1 (lowercase)
        if first_word2 in [word.lower() for word in words1]:
            matched = True
            break  # No need to check further for this element in unique_values_list

    # Increment counts based on match status and add items to lists
    if matched:
        matched_item_count_list2 += 1
        matched_items_list2.append(val2)
    else:
        unmatched_item_count_list2 += 1
        unmatched_items_list2.append(val2)

# Combine the counts and lists from both comparisons
total_matched_item_count_list = matched_item_count_list + matched_item_count_list2
total_unmatched_item_count_list = unmatched_item_count_list + unmatched_item_count_list2
total_matched_items_list = matched_items_list + matched_items_list2
total_unmatched_items_list = unmatched_items_list + unmatched_items_list2

# Print the counts and values from unique_values_list
print("Total count of matched items in unique_values_list:", total_matched_item_count_list)
print("Total count of unmatched items in unique_values_list:", total_unmatched_item_count_list)
# print("Matched items in unique_values_list:", total_matched_items_list)
# print("Unmatched items in unique_values_list:", total_unmatched_items_list)

# Create DataFrames for matched and unmatched items
matched_items_df = pd.DataFrame({'matched_items': total_matched_items_list})
unmatched_items_df = pd.DataFrame({'unmatched_items': total_unmatched_items_list})

# Specify the Excel file paths where you want to save the data
matched_excel_file_path = 'matched_output.xlsx'
unmatched_excel_file_path = 'unmatched_output.xlsx'

# Write the DataFrames to separate Excel files
matched_items_df.to_excel(matched_excel_file_path, index=False)
unmatched_items_df.to_excel(unmatched_excel_file_path, index=False)

▸,:,


ModuleNotFoundError: No module named 'docx'

In [8]:
#Run this!!!-Finalized Normalization code for Medication with cohort alone
import re
import docx
import csv
# Initialize counts for matched and unmatched items in unique_values_list
total_matched_item_count_list = 0
total_unmatched_item_count_list = 0
total_matched_items_list = []
total_unmatched_items_list = []

# Function to check if a word is numeric
def is_numeric(word):
    try:
        float(word)
        return True
    except ValueError:
        return False

# Iterate over elements in unique_values_list
for val1 in unique_values_list:
    # Split the words in val1 using space or special characters
    words1 = re.split(r'[ \W]', val1)
    # Take the first word from words1 (lowercase)
    first_word1 = words1[0].lower() if words1 else ""
    # print(first_word1)

    # Initialize a flag for matching against unique_values_list1
    matched1 = False

    # Iterate over elements in unique_values_list1
    for val2 in unique_values_list1:
        # Split the words in val2 using space or special characters
        words2 = re.split(r'[ \W]', val2)
        # print(words2)

        # Check if the first word from val1 (lowercase) matches any word from val2 (lowercase)
        if first_word1 in [word.lower() for word in words2]:
            matched1 = True
            break  # No need to check further for this element in unique_values_list1

    # Initialize a flag for matching against unique_values_list1
    matched2 = False

    # Iterate over elements in unique_values_list1
    for val2 in unique_values_list1:
        # Split the words in val2 using space or special characters
        words2 = re.split(r'[ \W]', val2)
        # Take the first word from words2 (lowercase)
        first_word2 = words2[0].lower() if words2 else ""
        # print(words2)

        # Check if the first word from val2 (lowercase) matches any word from val1 (lowercase)
        if first_word2 in [word.lower() for word in words1]:
            matched2 = True
            break  # No need to check further for this element in unique_values_list1

    # Increment counts and add val1 to lists based on match status
    if matched1 or matched2:
        total_matched_item_count_list += 1
        total_matched_items_list.append(val1)
    else:
        total_unmatched_item_count_list += 1
        total_unmatched_items_list.append(val1)

# Print the counts and values from unique_values_list
print("Total count of matched items in unique_values_list:", total_matched_item_count_list)
print("Total count of unmatched items in unique_values_list:", total_unmatched_item_count_list)
# print("Matched items in unique_values_list:", total_matched_items_list)
# print("Unmatched items in unique_values_list:", total_unmatched_items_list)

# # Create DataFrames for matched and unmatched items
# matched_items_df = pd.DataFrame({'matched_items': total_matched_items_list})
# unmatched_items_df = pd.DataFrame({'unmatched_items': total_unmatched_items_list})

# # Specify the Excel file paths where you want to save the data
# matched_excel_file_path = 'matched_output.xlsx'
# unmatched_excel_file_path = 'unmatched_output.xlsx'

# # Write the DataFrames to separate Excel files
# matched_items_df.to_excel(matched_excel_file_path, index=False)
# unmatched_items_df.to_excel(unmatched_excel_file_path, index=False)
# Save matched and unmatched items to a CSV file
with open('matched_items.csv', 'w', newline='') as csvfile:
    fieldnames = ['Matched Items']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

    writer.writeheader()
    for matched_item in total_matched_items_list:
        writer.writerow({'Matched Items': matched_item})

print("CSV file 'matched_items.csv' has been created with matched and unmatched items.")
with open('unmatched_items.csv', 'w', newline='') as csvfile:
    fieldnames = ['Unmatched Items']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

    writer.writeheader()
    for unmatched_item in total_unmatched_items_list:
        writer.writerow({'Unmatched Items': unmatched_item})

print("CSV file 'unmatched_items.csv' has been created with unmatched items.")

▸,:,


Total count of matched items in unique_values_list: 21955
Total count of unmatched items in unique_values_list: 674
CSV file 'matched_items.csv' has been created with matched and unmatched items.
CSV file 'unmatched_items.csv' has been created with unmatched items.


In [10]:
#Run this!!!-Finalized Normalization code for Medication for both together
import re
# import docx
import csv
# Initialize counts for matched and unmatched items in unique_values_list
total_matched_item_count_list = 0
total_unmatched_item_count_list = 0
total_matched_items_list = []
total_unmatched_items_list = []

# Function to check if a word is numeric
def is_numeric(word):
    try:
        float(word)
        return True
    except ValueError:
        return False

# Iterate over elements in unique_values_list
for val1 in unique_drugname_list_combined:
    # Split the words in val1 using space or special characters
    words1 = re.split(r'[ \W]', val1)
    # Take the first word from words1 (lowercase)
    first_word1 = words1[0].lower() if words1 else ""
    # print(first_word1)

    # Initialize a flag for matching against unique_values_list1
    matched1 = False

    # Iterate over elements in unique_values_list1
    for val2 in unique_values_list1:
        # Split the words in val2 using space or special characters
        words2 = re.split(r'[ \W]', val2)
        # print(words2)

        # Check if the first word from val1 (lowercase) matches any word from val2 (lowercase)
        if first_word1 in [word.lower() for word in words2]:
            matched1 = True
            break  # No need to check further for this element in unique_values_list1

    # Initialize a flag for matching against unique_values_list1
    matched2 = False

    # Iterate over elements in unique_values_list1
    for val2 in unique_values_list1:
        # Split the words in val2 using space or special characters
        words2 = re.split(r'[ \W]', val2)
        # Take the first word from words2 (lowercase)
        first_word2 = words2[0].lower() if words2 else ""
        # print(words2)

        # Check if the first word from val2 (lowercase) matches any word from val1 (lowercase)
        if first_word2 in [word.lower() for word in words1]:
            matched2 = True
            break  # No need to check further for this element in unique_values_list1

    # Increment counts and add val1 to lists based on match status
    if matched1 or matched2:
        total_matched_item_count_list += 1
        total_matched_items_list.append(val1)
    else:
        total_unmatched_item_count_list += 1
        total_unmatched_items_list.append(val1)

# Print the counts and values from unique_values_list
print("Total count of matched items in unique_values_list:", total_matched_item_count_list)
print("Total count of unmatched items in unique_values_list:", total_unmatched_item_count_list)
# print("Matched items in unique_values_list:", total_matched_items_list)
# print("Unmatched items in unique_values_list:", total_unmatched_items_list)

# # Create DataFrames for matched and unmatched items
# matched_items_df = pd.DataFrame({'matched_items': total_matched_items_list})
# unmatched_items_df = pd.DataFrame({'unmatched_items': total_unmatched_items_list})

# # Specify the Excel file paths where you want to save the data
# matched_excel_file_path = 'matched_output.xlsx'
# unmatched_excel_file_path = 'unmatched_output.xlsx'

# # Write the DataFrames to separate Excel files
# matched_items_df.to_excel(matched_excel_file_path, index=False)
# unmatched_items_df.to_excel(unmatched_excel_file_path, index=False)
# Save matched and unmatched items to a CSV file
with open('matched_items_latest.csv', 'w', newline='') as csvfile:
    fieldnames = ['Matched Items']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

    writer.writeheader()
    for matched_item in total_matched_items_list:
        writer.writerow({'Matched Items': matched_item})

print("CSV file 'matched_items_latest.csv' has been created with matched and unmatched items.")
with open('unmatched_items_latest.csv', 'w', newline='') as csvfile:
    fieldnames = ['Unmatched Items']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

    writer.writeheader()
    for unmatched_item in total_unmatched_items_list:
        writer.writerow({'Unmatched Items': unmatched_item})

print("CSV file 'unmatched_items_latest.csv' has been created with unmatched items.")

▸,:,


NameError: name 'unique_drugname_list_combined' is not defined

In [ ]:
import re
import csv

# Initialize counts for matched and unmatched items
matched_item_count = 0
unmatched_item_count = 0

# Lists to store matched and unmatched items
matched_items = []
unmatched_items = []

# Function to check if a word is numeric
def is_numeric(word):
    try:
        float(word)
        return True
    except ValueError:
        return False

# Iterate over elements in unique_values_list
for val1 in unique_values_list:
    # Split the words in val1 using space or special characters
    words1 = re.split(r'[ \W]', val1)
    # Take the first word from words1 (lowercase)
    first_word1 = words1[0].lower() if words1 else ""
    matched = False  # Flag to check if any match is found
    
    # Iterate over elements in unique_values_list1
    for val2 in unique_values_list1:
        # Split the words in val2 using space or special characters
        words2 = re.split(r'[ \W]', val2)
        
        # Check if the first word from val1 (lowercase) matches any word from val2 (lowercase)
        if first_word1 in [word.lower() for word in words2]:
            matched = True
            break  # No need to check further for this element in unique_values_list1 

    # Increment counts based on match status and add items to lists
    if matched:
        matched_item_count += 1
        matched_items.append(val1)
    else:
        unmatched_item_count += 1
        unmatched_items.append(val1)

# Iterate over elements in unique_values_list1 to check vice versa
for val2 in unique_values_list1:
    # Split the words in val2 using space or special characters
    words2 = re.split(r'[ \W]', val2)
    # Take the first word from words2 (lowercase)
    first_word2 = words2[0].lower() if words2 else ""
    matched = False  # Flag to check if any match is found
    
    # Iterate over elements in unique_values_list
    for val1 in unique_values_list:
        # Split the words in val1 using space or special characters
        words1 = re.split(r'[ \W]', val1)
        
        # Check if the first word from val2 (lowercase) matches any word from val1 (lowercase)
        if first_word2 in [word.lower() for word in words1]:
            matched = True
            break  # No need to check further for this element in unique_values_list

    # Increment counts based on match status and add items to lists
    if matched:
        matched_item_count += 1
        matched_items.append(val2)
    else:
        unmatched_item_count += 1
        unmatched_items.append(val2)

# Print the counts of matched and unmatched items
print("Total count of items with at least one word match:", matched_item_count)
print("Total count of unmatched items:", unmatched_item_count)

In [ ]:
import re
import csv

# Initialize counts for matched and unmatched items
matched_item_count = 0
unmatched_item_count = 0

# Lists to store matched and unmatched items
matched_items = []
unmatched_items = []

matched_items_list1 = []
unmatched_items_list1 = []

# Function to check if a word is numeric
def is_numeric(word):
    try:
        float(word)
        return True
    except ValueError:
        return False

# Iterate over elements in unique_values_list
for val1 in unique_values_list:
    # Split the words in val1 using space or special characters
    words1 = re.split(r'[ \W]', val1)
    # Take the first word from words1 (lowercase)
    first_word1 = words1[0].lower() if words1 else ""
    matched = False  # Flag to check if any match is found
    
    # Iterate over elements in unique_values_list1
    for val2 in unique_values_list1:
        # Split the words in val2 using space or special characters
        words2 = re.split(r'[ \W]', val2)
        
        # Check if the first word from val1 (lowercase) matches any word from val2 (lowercase)
        if first_word1 in [word.lower() for word in words2]:
            matched = True
            break  # No need to check further for this element in unique_values_list1 

    # Increment counts based on match status and add items to lists
    if matched:
        matched_item_count += 1
        matched_items.append(val1)
    else:
        unmatched_item_count += 1
        unmatched_items.append(val1)

# Iterate over elements in unique_values_list1 to check vice versa
for val2 in unique_values_list1:
    # Split the words in val2 using space or special characters
    words2 = re.split(r'[ \W]', val2)
    # Take the first word from words2 (lowercase)
    first_word2 = words2[0].lower() if words2 else ""
    matched = False  # Flag to check if any match is found
    
    # Iterate over elements in unique_values_list
    for val1 in unique_values_list:
        # Split the words in val1 using space or special characters
        words1 = re.split(r'[ \W]', val1)
        
        # Check if the first word from val2 (lowercase) matches any word from val1 (lowercase)
        if first_word2 in [word.lower() for word in words1]:
            matched = True
            break  # No need to check further for this element in unique_values_list

    # Increment counts based on match status and add items to lists
    if matched:
        matched_item_count += 1
        matched_items_list1.append(val2)
    else:
        unmatched_item_count += 1
        unmatched_items_list1.append(val2)

# Print the counts of matched and unmatched items
print("Total count of items with at least one word match:", matched_item_count)
print("Total count of unmatched items:", unmatched_item_count)

# Print the matched and unmatched items
print("Matched items from unique_values_list:", matched_items)
print("Unmatched items from unique_values_list:", unmatched_items)
print("Matched items from unique_values_list1:", matched_items_list1)
print("Unmatched items from unique_values_list1:", unmatched_items_list1)

In [ ]:
import re

# Initialize counts for matched and unmatched items in unique_values_list
matched_item_count_list = 0
unmatched_item_count_list = 0
matched_items_list = []
unmatched_items_list = []

# Function to check if a word is numeric
def is_numeric(word):
    try:
        float(word)
        return True
    except ValueError:
        return False

# Iterate over elements in unique_values_list
for val1 in unique_values_list:
    # Split the words in val1 using space or special characters
    words1 = re.split(r'[ \W]', val1)
    # Take the first word from words1 (lowercase)
    first_word1 = words1[0].lower() if words1 else ""
    matched = False  # Flag to check if any match is found
    
    # Iterate over elements in unique_values_list1
    for val2 in unique_values_list1:
        # Split the words in val2 using space or special characters
        words2 = re.split(r'[ \W]', val2)
        
        # Check if the first word from val1 (lowercase) matches any word from val2 (lowercase)
        if first_word1 in [word.lower() for word in words2]:
            matched = True
            break  # No need to check further for this element in unique_values_list1 

    # Increment counts based on match status and add items to lists
    if matched:
        matched_item_count_list += 1
        matched_items_list.append(val1)
    else:
        unmatched_item_count_list += 1
        unmatched_items_list.append(val1)

# Reset counts for matched and unmatched items
matched_item_count_list2 = 0
unmatched_item_count_list2 = 0
matched_items_list2 = []
unmatched_items_list2 = []

# Iterate over elements in unique_values_list1 and compare them to unique_values_list
for val2 in unique_values_list1:
    # Split the words in val2 using space or special characters
    words2 = re.split(r'[ \W]', val2)
    # Take the first word from words2 (lowercase)
    first_word2 = words2[0].lower() if words2 else ""
    matched = False  # Flag to check if any match is found
    
    # Iterate over elements in unique_values_list
    for val1 in unique_values_list:
        # Split the words in val1 using space or special characters
        words1 = re.split(r'[ \W]', val1)
        
        # Check if the first word from val2 (lowercase) matches any word from val1 (lowercase)
        if first_word2 in [word.lower() for word in words1]:
            matched = True
            break  # No need to check further for this element in unique_values_list

    # Increment counts based on match status and add items to lists
    if matched:
        matched_item_count_list2 += 1
        matched_items_list2.append(val2)
    else:
        unmatched_item_count_list2 += 1
        unmatched_items_list2.append(val2)

# Combine the counts and lists from both comparisons
total_matched_item_count_list = matched_item_count_list + matched_item_count_list2
total_unmatched_item_count_list = unmatched_item_count_list + unmatched_item_count_list2
total_matched_items_list = matched_items_list + matched_items_list2
total_unmatched_items_list = unmatched_items_list + unmatched_items_list2

# Print the counts and values from unique_values_list
print("Total count of matched items in unique_values_list:", total_matched_item_count_list)
print("Total count of unmatched items in unique_values_list:", total_unmatched_item_count_list)
print("Matched items in unique_values_list:", total_matched_items_list)
print("Unmatched items in unique_values_list:", total_unmatched_items_list)

In [ ]:
# # Initialize a count for matched items
# matched_item_count = 0

# # Iterate over elements in unique_values_list
# for val1 in unique_values_list:
#     words1 = val1.split()
    
#     # Iterate over elements in unique_values_list1
#     for val2 in unique_values_list1:
#         words2 = val2.split()
        
#         # Check if there is at least one word match
#         if any(word1 == word2 for word1 in words1 for word2 in words2):
#             matched_item_count += 1
#             break  # No need to check further for this element in unique_values_list1

# # Print the total count of matched items
# print("Total count of items with at least one word match:", matched_item_count)
# Initialize counts for matched and unmatched items
matched_item_count = 0
unmatched_item_count = 0

# Iterate over elements in unique_values_list
for val1 in unique_values_list:
    words1 = val1.split()
    matched = False  # Flag to check if any match is found
    
    # Iterate over elements in unique_values_list1
    for val2 in unique_values_list1:
        words2 = val2.split()
        
        # Check if there is at least one word match
        if any(word1 == word2 for word1 in words1 for word2 in words2):
            matched = True
            break  # No need to check further for this element in unique_values_list1
    
    # Increment counts based on match status
    if matched:
        matched_item_count += 1
    else:
        unmatched_item_count += 1

# Print the counts of matched and unmatched items
print("Total count of items with at least one word match:", matched_item_count)
print("Total count of unmatched items:", unmatched_item_count)


In [ ]:
import re
import csv

# Initialize counts for matched and unmatched items
matched_item_count = 0
unmatched_item_count = 0

# Lists to store matched and unmatched items
matched_items = []
unmatched_items = []

# Iterate over elements in unique_values_list
for val1 in unique_values_list:
    # Split the words in val1 using space or special characters
    words1 = re.split(r'[ \W]', val1)
    # Take the first word from words1 (lowercase)
    first_word1 = words1[0].lower() if words1 else ""
    matched = False  # Flag to check if any match is found
    
    # Iterate over elements in unique_values_list1
    for val2 in unique_values_list1:
        # Split the words in val2 using space or special characters
        words2 = re.split(r'[ \W]', val2)
        
        # Check if the first word from val1 (lowercase) matches any word from val2 (lowercase)
        if first_word1 in [word.lower() for word in words2]:
            matched = True
            break  # No need to check further for this element in unique_values_list1
    
    # Increment counts based on match status and add items to lists
    if matched:
        matched_item_count += 1
        matched_items.append(val1)
    else:
        unmatched_item_count += 1
        unmatched_items.append(val1)

# Print the counts of matched and unmatched items
print("Total count of items with at least one word match:", matched_item_count)
print("Total count of unmatched items:", unmatched_item_count)

# Save matched and unmatched items to a CSV file
with open('matched_unmatched_items.csv', 'w', newline='') as csvfile:
    fieldnames = ['Matched Items', 'Unmatched Items']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

    writer.writeheader()
    for matched_item, unmatched_item in zip(matched_items, unmatched_items):
        writer.writerow({'Matched Items': matched_item, 'Unmatched Items': unmatched_item})

print("CSV file 'matched_unmatched_items.csv' has been created with matched and unmatched items.")

In [ ]:
# import re  # Import the re module for regular expressions
# # Initialize counts for matched and unmatched items
# matched_item_count = 0
# unmatched_item_count = 0

# # Iterate over elements in unique_values_list
# for val1 in unique_values_list:
#     # Split the words in val1 using space or special characters
#     words1 = re.split(r'[ \W]', val1)
#     # Take the first word from words1
#     first_word1 = words1[0] if words1 else ""
#     matched = False  # Flag to check if any match is found
    
#     # Iterate over elements in unique_values_list1
#     for val2 in unique_values_list1:
#         # Split the words in val2 using space or special characters
#         words2 = re.split(r'[ \W]', val2)
        
#         # Check if the first word from val1 matches any word from val2
#         if first_word1 in words2:
#             matched = True
#             break  # No need to check further for this element in unique_values_list1
    
#     # Increment counts based on match status
#     if matched:
#         matched_item_count += 1
#     else:
#         unmatched_item_count += 1

# # Print the counts of matched and unmatched items
# print("Total count of items with at least one word match:", matched_item_count)
# print("Total count of unmatched items:", unmatched_item_count)
# import re  # Import the re module for regular expressions
# import csv

# # Initialize counts for matched and unmatched items
# matched_item_count = 0
# unmatched_item_count = 0

# # Lists to store matched and unmatched items
# matched_items = []
# unmatched_items = []

# # Iterate over elements in unique_values_list
# for val1 in unique_values_list:
#     # Split the words in val1 using space or special characters
#     words1 = re.split(r'[ \W]', val1)
#     # Take the first word from words1
#     first_word1 = words1[0] if words1 else ""
#     matched = False  # Flag to check if any match is found
    
#     # Iterate over elements in unique_values_list1
#     for val2 in unique_values_list1:
#         # Split the words in val2 using space or special characters
#         words2 = re.split(r'[ \W]', val2)
        
#         # Check if the first word from val1 matches any word from val2
#         if first_word1 in words2:
#             matched = True
#             break  # No need to check further for this element in unique_values_list1
    
#     # Increment counts based on match status and add items to lists
#     if matched:
#         matched_item_count += 1
#         matched_items.append(val1)
#     else:
#         unmatched_item_count += 1
#         unmatched_items.append(val1)

# # Print the counts of matched and unmatched items
# print("Total count of items with at least one word match:", matched_item_count)
# print("Total count of unmatched items:", unmatched_item_count)

# # Save matched and unmatched items to a CSV file
# with open('matched_unmatched_items.csv', 'w', newline='') as csvfile:
#     fieldnames = ['Matched Items', 'Unmatched Items']
#     writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

#     writer.writeheader()
#     for matched_item, unmatched_item in zip(matched_items, unmatched_items):
#         writer.writerow({'Matched Items': matched_item, 'Unmatched Items': unmatched_item})

# print("CSV file 'matched_unmatched_items.csv' has been created with matched and unmatched items.")
import re
import csv

# Initialize counts for matched and unmatched items
matched_item_count = 0
unmatched_item_count = 0

# Lists to store matched and unmatched items
matched_items = []
unmatched_items = []

# Iterate over elements in unique_values_list
for val1 in unique_values_list:
    # Split the words in val1 using space or special characters
    words1 = re.split(r'[ \W]', val1)
    # Take the first word from words1
    first_word1 = words1[0].lower() if words1 else ""
    matched = False  # Flag to check if any match is found
    
    # Iterate over elements in unique_values_list1
    for val2 in unique_values_list1:
        # Split the words in val2 using space or special characters
        words2 = re.split(r'[ \W]', val2)
        
        # Check if the first word from val1 (lowercase) matches any word from val2 (lowercase)
        if first_word1 in [word.lower() for word in words2]:
            matched = True
            break  # No need to check further for this element in unique_values_list1
    
    # Increment counts based on match status and add items to lists
    if matched:
        matched_item_count += 1
        matched_items.append(val1)
    else:
        unmatched_item_count += 1
        unmatched_items.append(val1)

# Print the counts of matched and unmatched items
print("Total count of items with at least one word match:", matched_item_count)
print("Total count of unmatched items:", unmatched_item_count)

# Save matched and unmatched items to a CSV file
with open('matched_unmatched_items.csv', 'w', newline='') as csvfile:
    fieldnames = ['Matched Items', 'Unmatched Items']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

    writer.writeheader()
    for matched_item, unmatched_item in zip(matched_items, unmatched_items):
        writer.writerow({'Matched Items': matched_item, 'Unmatched Items': unmatched_item})

print("CSV file 'matched_unmatched_items.csv' has been created with matched and unmatched items.")

In [ ]:
import re
import csv

# Initialize counts for matched and unmatched items
matched_item_count = 0
unmatched_item_count = 0

# Lists to store matched and unmatched items
matched_items = []
unmatched_items = []

# Iterate over elements in unique_values_list
for val1 in unique_values_list:
    # Split the words in val1 using space or special characters
    words1 = re.split(r'[ \W]', val1)
    # Take the first word from words1 (lowercase)
    first_word1 = words1[0].lower() if words1 else ""
    matched = False  # Flag to check if any match is found
    
    # Iterate over elements in unique_values_list1
    for val2 in unique_values_list1:
        # Split the words in val2 using space or special characters
        words2 = re.split(r'[ \W]', val2)
        
        # Check if the first word from val1 (lowercase) matches any word from val2 (lowercase)
        if first_word1 in [word.lower() for word in words2]:
            matched = True
            break  # No need to check further for this element in unique_values_list1
    
    # Check if all words in val1 (except the first word) exist in any form (case-insensitive) within val2
    if not matched:
        for word1 in words1[1:]:
            if any(word1.lower() in word2.lower() for word2 in words2):
                matched = True
                break
    
    # Increment counts based on match status and add items to lists
    if matched:
        matched_item_count += 1
        matched_items.append(val1)
    else:
        unmatched_item_count += 1
        unmatched_items.append(val1)

# Print the counts of matched and unmatched items
print("Total count of items with at least one word match:", matched_item_count)
print("Total count of unmatched items:", unmatched_item_count)

# Save matched and unmatched items to a CSV file
with open('matched_unmatched_itemsv1.csv', 'w', newline='') as csvfile:
    fieldnames = ['Matched Items', 'Unmatched Items']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

    writer.writeheader()
    for matched_item, unmatched_item in zip(matched_items, unmatched_items):
        writer.writerow({'Matched Items': matched_item, 'Unmatched Items': unmatched_item})

print("CSV file 'matched_unmatched_items.csv' has been created with matched and unmatched items.")

In [ ]:
# import re
# import csv

# # Initialize counts for matched and unmatched items
# matched_item_count = 0
# unmatched_item_count = 0

# # Lists to store matched and unmatched items
# matched_items = []
# unmatched_items = []

# # Function to check if a word is numeric
# def is_numeric(word):
#     try:
#         float(word)
#         return True
#     except ValueError:
#         return False

# # Iterate over elements in unique_values_list
# for val1 in unique_values_list:
#     # Split the words in val1 using space or special characters
#     words1 = re.split(r'[ \W]', val1)
#     # Take the first word from words1 (lowercase)
#     first_word1 = words1[0].lower() if words1 else ""
#     matched = False  # Flag to check if any match is found
    
#     # Iterate over elements in unique_values_list1
#     for val2 in unique_values_list1:
#         # Split the words in val2 using space or special characters
#         words2 = re.split(r'[ \W]', val2)
        
#         # Check if the first word from val1 (lowercase) matches any word from val2 (lowercase)
#         if first_word1 in [word.lower() for word in words2]:
#             matched = True
#             break  # No need to check further for this element in unique_values_list1
    
#     # Check if all non-numeric words in val1 (except the first word) exist in any form (case-insensitive) within val2
#     if not matched:
#         for word1 in words1[1:]:
#             if not is_numeric(word1) and any(word1.lower() in word2.lower() for word2 in words2):
#                 matched = True
#                 break
    
#     # Increment counts based on match status and add items to lists
#     if matched:
#         matched_item_count += 1
#         matched_items.append(val1)
#     else:
#         unmatched_item_count += 1
#         unmatched_items.append(val1)

# # Print the counts of matched and unmatched items
# print("Total count of items with at least one word match:", matched_item_count)
# print("Total count of unmatched items:", unmatched_item_count)

# # Save matched and unmatched items to a CSV file
# with open('matched_unmatched_itemsv1.csv', 'w', newline='') as csvfile:
#     fieldnames = ['Matched Items', 'Unmatched Items']
#     writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

#     writer.writeheader()
#     for matched_item, unmatched_item in zip(matched_items, unmatched_items):
#         writer.writerow({'Matched Items': matched_item, 'Unmatched Items': unmatched_item})

# print("CSV file 'matched_unmatched_items.csv' has been created with matched and unmatched items.")
import re
import csv

# Initialize counts for matched and unmatched items
matched_item_count = 0
unmatched_item_count = 0

# Lists to store matched and unmatched items
matched_items = []
unmatched_items = []

# Function to check if a word is numeric
def is_numeric(word):
    try:
        float(word)
        return True
    except ValueError:
        return False

# Iterate over elements in unique_values_list
for val1 in unique_values_list:
    # Split the words in val1 using space or special characters
    words1 = re.split(r'[ \W]', val1)
    # Take the first word from words1 (lowercase)
    first_word1 = words1[0].lower() if words1 else ""
    matched = False  # Flag to check if any match is found
    
    # Iterate over elements in unique_values_list1
    for val2 in unique_values_list1:
        # Split the words in val2 using space or special characters
        words2 = re.split(r'[ \W]', val2)
        
        # Check if the first word from val1 (lowercase) matches any word from val2 (lowercase)
        if first_word1 in [word.lower() for word in words2]:
            matched = True
            break  # No need to check further for this element in unique_values_list1
    
    # Check if the first non-numeric word in val1 exists in any form (case-insensitive) within any value from unique_values_list
    if not matched:
        first_non_numeric_word = next((word1.lower() for word1 in words1[1:] if not is_numeric(word1)), None)
        if first_non_numeric_word:
            for val2 in unique_values_list:
                words2 = re.split(r'[ \W]', val2)
                if any(first_non_numeric_word in word2.lower() for word2 in words2):
                    matched = True
                    break
    
    # Increment counts based on match status and add items to lists
    if matched:
        matched_item_count += 1
        matched_items.append(val1)
    else:
        unmatched_item_count += 1
        unmatched_items.append(val1)

# Print the counts of matched and unmatched items
print("Total count of items with at least one word match:", matched_item_count)
print("Total count of unmatched items:", unmatched_item_count)

# # Print matched and unmatched items as lists
print("Matched Items:", matched_items)
for item in matched_items:
    print(item)

print("\nUnmatched Items:")
for item in unmatched_items:
    print(item)

# Save matched and unmatched items to a CSV file
with open('matched_unmatched_itemsv1.csv', 'w', newline='') as csvfile:
    fieldnames = ['Matched Items', 'Unmatched Items']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

    writer.writeheader()
    for matched_item, unmatched_item in zip(matched_items, unmatched_items):
        writer.writerow({'Matched Items': matched_item, 'Unmatched Items': unmatched_item})

print("CSV file 'matched_unmatched_items.csv' has been created with matched and unmatched items.")

In [ ]:
import re
import csv

# Initialize counts for matched and unmatched items
matched_item_count = 0
unmatched_item_count = 0

# Lists to store matched and unmatched items
matched_items = []
unmatched_items = []

# Function to check if a word is numeric
def is_numeric(word):
    try:
        float(word)
        return True
    except ValueError:
        return False

# Iterate over elements in unique_values_list
for val1 in unique_values_list:
    # Split the words in val1 using space or special characters
    words1 = re.split(r'[ \W]', val1)
    # Take the first word from words1 (lowercase)
    first_word1 = words1[0].lower() if words1 else ""
    matched = False  # Flag to check if any match is found
    
    # Iterate over elements in unique_values_list1
    for val2 in unique_values_list1:
        # Split the words in val2 using space or special characters
        words2 = re.split(r'[ \W]', val2)
        
        # Check if the first word from val1 (lowercase) matches any word from val2 (lowercase)
        if first_word1 in [word.lower() for word in words2]:
            matched = True
            break  # No need to check further for this element in unique_values_list1
    
    # Check if the first non-numeric word in val1 exists in any form (case-insensitive) within any value from unique_values_list1
    if not matched:
        first_non_numeric_word = next((word1.lower() for word1 in words1[1:] if not is_numeric(word1)), None)
        if first_non_numeric_word:
            for val2 in unique_values_list1:
                words2 = re.split(r'[ \W]', val2)
                if any(first_non_numeric_word in word2.lower() for word2 in words2) or \
                   any(first_non_numeric_word.lower() in word2.lower() for word2 in words2 if not is_numeric(word2)):
                    matched = True
                    break
    
    # Check if the first non-numeric word in val2 exists in any form (case-insensitive) within any value from unique_values_list
    if not matched:
        first_word2 = words2[0].lower() if words2 else ""
        first_non_numeric_word2 = next((word2.lower() for word2 in words2[1:] if not is_numeric(word2)), None)
        if first_non_numeric_word2:
            for val1 in unique_values_list:
                words1 = re.split(r'[ \W]', val1)
                if (first_non_numeric_word2 in [word1.lower() for word1 in words1] or
                    any(first_non_numeric_word2.lower() in word1.lower() for word1 in words1 if not is_numeric(word1))) and first_word1 != first_word2:
                    matched = True
                    break
    
    # Increment counts based on match status and add items to lists
    if matched:
        matched_item_count += 1
        matched_items.append(val1)
    else:
        unmatched_item_count += 1
        unmatched_items.append(val1)

# Print the counts of matched and unmatched items
print("Total count of items with at least one word match:", matched_item_count)
print("Total count of unmatched items:", unmatched_item_count)

# Print matched and unmatched items as lists
print("Matched Items:")
for item in matched_items:
    print(item)

print("\nUnmatched Items:")
for item in unmatched_items:
    print(item)

# Save matched and unmatched items to a CSV file
with open('matched_unmatched_itemsv1.csv', 'w', newline='') as csvfile:
    fieldnames = ['Matched Items', 'Unmatched Items']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

    writer.writeheader()
    for matched_item, unmatched_item in zip(matched_items, unmatched_items):
        writer.writerow({'Matched Items': matched_item, 'Unmatched Items': unmatched_item})

print("CSV file 'matched_unmatched_items.csv' has been created with matched and unmatched items.")

In [ ]:
import pandas as pd

# Replace 'your_file.csv' with the path to your CSV file
csv_file = 'matched_unmatched_itemsv1.csv'

# Read the CSV file into a Pandas DataFrame
pddf = pd.read_csv(csv_file)

display(pddf)

# Now 'df' contains your data as a DataFrame, and you can perform various operations on it.


In [ ]:
# Initialize a count for matched items
matched_item_count = 0

# Initialize a list to store the updated items
updated_values_list = []

# Iterate over elements in unique_values_list
for val1 in unique_values_list:
    words1 = val1.split()
    matched = False  # Flag to check if any match is found
    
    # Iterate over elements in unique_values_list1
    for val2 in unique_values_list1:
        words2 = val2.split()
        
        # Check if there is at least one word match
        if any(word1 == word2 for word1 in words1 for word2 in words2):
            matched = True
            updated_values_list.append(val2)  # Replace with the matching item
            break  # No need to check further for this element in unique_values_list1
    
    # Increment counts based on match status
    if matched:
        matched_item_count += 1
    else:
        updated_values_list.append(val1)  # Keep the original item

# Print the counts of matched and unmatched items
print("Total count of items with at least one word match:", matched_item_count)

# Print the updated list
print("Updated List:")
for updated_val in updated_values_list:
    print(updated_val)

In [ ]:
import csv
# Initialize a count for matched items
matched_item_count = 0

# Initialize a list to store the updated items
updated_values_list = []

# Create a CSV file for output
with open('matched_items.csv', 'w', newline='') as csvfile:
    fieldnames = ['Original Item', 'Replaced Item']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    
    writer.writeheader()  # Write CSV header
    
    # Iterate over elements in unique_values_list
    for val1 in unique_values_list:
        words1 = val1.split()
        matched = False  # Flag to check if any match is found
        replaced_item = None  # Store the replaced item
        
        # Iterate over elements in unique_values_list1
        for val2 in unique_values_list1:
            words2 = val2.split()
            
            # Check if there is at least one word match
            if any(word1 == word2 for word1 in words1 for word2 in words2):
                matched = True
                replaced_item = val2  # Replace with the matching item
                break  # No need to check further for this element in unique_values_list1
        
        # Increment counts based on match status and write to CSV
        if matched:
            matched_item_count += 1
            writer.writerow({'Original Item': val1, 'Replaced Item': replaced_item})
        else:
            updated_values_list.append(val1)  # Keep the original item

# Print the counts of matched and unmatched items
print("Total count of items with at least one word match:", matched_item_count)

# Print the updated list
print("Updated List:")
for updated_val in updated_values_list:
    print(updated_val)

print("CSV file 'matched_items.csv' has been created with original and replaced values.")

In [ ]:
# Compare the lists and find unmatched items
unmatched_items = [item for item in updated_values_list if item not in unique_values_list1]

# Count the unmatched items
unmatched_count = len(unmatched_items)

# Print the unmatched items and their count
print("Unmatched items:", unmatched_items)
print("Count of unmatched items:", unmatched_count)